### Preprocessing and augmentations ###

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import glob
import copy
import tarfile
from pathlib import Path
import cv2
from matplotlib import pyplot as plt

# Appearance of the Notebook
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Import this module with autoreload
%load_ext autoreload
%autoreload 2

import computervision as cv
from computervision.fileutils import FileOP
from computervision.imageproc import ImageData, is_image
from computervision.imageproc import plot_boxes, enclosing_box, xyxy2xywh, clipxywh, crop_image

from computervision.performance import DetectionMetrics

# Print version info
print(f'Package version: {cv.__version__}')
print(f'Authors:         {cv.__authors__}')
print(f'Python version:  {sys.version}')

Package version: v0.0.2
Authors:         The Core for Computational Biomedicine at Harvard Medical School
https://dbmi.hms.harvard.edu/about-dbmi/core-computational-biomedicine
Python version:  3.12.3 (main, Jun 18 2025, 17:59:45) [GCC 13.3.0]


In [2]:
data_dir = os.environ.get('DATA_DIR')
print(f'data_dir: {data_dir}')
# Directory to store the dentex data set
dataset_name = 'dataset_dental_roboflow'
dataset_dir = os.path.join(data_dir, 'roboflow')
image_dir = os.path.join(dataset_dir, dataset_name, 'cropped')

data_dir: /app/data


# Open the annotation file (in the image directory after extraction)
annotations_file_name = 'annotations_cropped_dset.parquet'
annotations_file = os.path.join(image_dir, annotations_file_name)
data_df = pd.read_parquet(annotations_file)
display(data_df.head(2))
print(data_df.shape)

file_col = 'file_name'
bbox_col= 'bbox'
label_col = 'label'

# Chceck the images
file_list = [os.path.join(image_dir, file_name) for file_name in data_df[file_col].unique()]
checked = [is_image(file) for file in file_list]
assert len(file_list) == sum(checked), f'WARNING: Could not open all {len(file_list)} images at: {image_dir}'
print(f'Image directory:        {image_dir}')
print(f'Total number of images: {len(file_list)}')
print(f'Annotations:            {data_df.shape[0]}')

In [4]:
# Image numbers
label_list = sorted(list(data_df[label_col].unique()))
for dset in ['train', 'val', 'test']:
    data_df_dset = data_df.loc[data_df['dset'] == dset]
    print()
    print(f'{len(data_df_dset[file_col].unique())} total images in dataset {dset.upper()}')
    n_label = data_df_dset[[label_col, file_col]].\
        groupby(label_col).\
        nunique().\
        reset_index(drop=False).\
        rename(columns={file_col: 'n_images'}).\
        sort_values(by='n_images', ascending=False).\
        reset_index(drop=True)
    display(n_label)


8308 total images in dataset TRAIN


,label,n_images
0,tooth,5011
1,composite,1430
2,amalgam,600
3,Calculus,567
4,caries,523
5,root filling,177



180 total images in dataset VAL


,label,n_images
0,Calculus,30
1,amalgam,30
2,caries,30
3,composite,30
4,root filling,30
5,tooth,30



180 total images in dataset TEST


,label,n_images
0,Calculus,30
1,amalgam,30
2,caries,30
3,composite,30
4,root filling,30
5,tooth,30


### Preprocessing of the images ###
All of the images have different sizes. For the model we need all images to be the same shape. It is not always a good idea to change the aspect ratio of the images and we also want to keep the size of the teeth relative to the whole image size. This is the process:

1. Find the largest image dimension and size S for that dimension in the entire data set
2. Scale all smaller images (with original aspect ration maintained) so that their largest dimension is S
3. Pad the smaller dimension to get square images
4. Scale the square image to the input size of the model

In [6]:
# We have the images sizes in the data frame
# Finding the largest image and its dimensions is easy
im_max_width = data_df['width'].max()
im_max_height = data_df['height'].max()
print(f'Maximum image width across the data set:  {im_max_width}')
print(f'Maximum image height across the data set: {im_max_height}')
# Set the maximum image size for all images
im_max_size = int(np.max((im_max_width, im_max_height)))
print(f'Maximum image size: {im_max_size}')

Maximum image width across the data set:  640
Maximum image height across the data set: 480
Maximum image size: 640


In [9]:
# Take a look at a few images
np.random.seed(234)
image_list = np.random.choice(data_df[file_col].unique(), size=5, replace=False)
for i, image in enumerate(image_list):
    # Load the image
    image_file = os.path.join(image_dir, image)
    im = ImageData().load_image(image_file)
    # Pre-process the image
    im_processed = load_and_process_image(image_file_path=image_file,
                                          max_image_size=im_max_size)
    # Show the two images side-by-side
    im_list = [im, im_processed]
    fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(6, 2))
    for a, axa in enumerate(ax):
        axa.imshow(im_list[a])
        axa.set_title(f'Image shape {im_list[a].shape}')
        axa.set(xticks=[], yticks=[])
    plt.show()

NameError: name 'load_and_process_image' is not defined